# OpenCV Optical Flow Tutorial: From 0 to Mastery

This notebook follows the style of `OpenCV_MeanShift_CamShift_From_Zero_to_Master_ATDL_Style.ipynb`: theory first, runnable local demos second, then debugging rules, practice tasks, and references.

本 notebook 用“从 0 到精通”的路线讲清楚 Optical Flow / 光流。你会学到：

- 光流到底在估计什么
- Brightness constancy / 亮度恒定假设为什么重要
- Lucas-Kanade sparse optical flow 如何追踪角点
- Farneback dense optical flow 如何为每个像素估计运动
- 金字塔、多尺度、窗口大小、特征质量如何影响结果
- 如何检查漂移、丢点、遮挡和大位移问题
- 如何把光流放进真实视频或摄像头循环

Core mindset:

`optical flow = match visual evidence between nearby frames + reject unreliable motion`

## How to Use This Notebook

Recommended learning order:

1. Read Part 1 to build the mental model.
2. Run Part 2 cell by cell and inspect every visualization.
3. Modify object speed, texture, noise, feature parameters, and flow parameters.
4. Use Part 3 as a debugging checklist and project roadmap.

Suggested environment:

- Python 3.10+
- `opencv-python`
- `numpy`
- `matplotlib`

## Part 1: Theoretical Foundations

### 1.1 What Problem Is Optical Flow Solving?

In video, many tasks need to know how image content moves from one frame to the next.

Typical questions:

1. Where did this corner move?
2. Which pixels are moving together?
3. How fast is the camera or object moving?
4. Is there motion that looks unusual?
5. Can I stabilize, segment, or track using only frame-to-frame motion?

The key idea:

`If a visible point belongs to the same physical surface, its image brightness pattern should look similar in the next frame.`

Optical flow returns a 2D motion vector:

`flow(x, y) = (u, v)`

where `u` is horizontal displacement and `v` is vertical displacement in pixels.

### 1.2 Brightness Constancy and the Aperture Problem

The classical optical flow assumption is brightness constancy:

`I(x, y, t) ~= I(x + u, y + v, t + 1)`

For small motion, this leads to the optical flow constraint equation:

`Ix * u + Iy * v + It = 0`

One equation has two unknowns, so a single pixel is not enough. This is the aperture problem.

Practical algorithms add extra assumptions:

- **Lucas-Kanade**: nearby pixels inside a small window share the same motion.
- **Farneback**: local neighborhoods can be approximated by polynomial surfaces.
- **Pyramids**: large motion becomes smaller when images are downsampled.

Master-level habit:

`Flow is trustworthy where texture is rich, motion is small enough, and the visual pattern remains visible.`

### 1.3 Sparse Optical Flow: Lucas-Kanade

Sparse optical flow tracks selected points, usually corners.

Typical pipeline:

1. Convert frame 0 to grayscale.
2. Detect good feature points using `cv2.goodFeaturesToTrack`.
3. Track them with `cv2.calcOpticalFlowPyrLK`.
4. Keep points whose status flag is good.
5. Draw tracks or use point motion for higher-level logic.

OpenCV API:

```python
next_pts, status, error = cv2.calcOpticalFlowPyrLK(prev_gray, next_gray, prev_pts, None, **lk_params)
```

Useful when:

- you need object/camera motion cues
- you want fast tracking
- you only care about reliable corners, not every pixel

### 1.4 Dense Optical Flow: Farneback

Dense optical flow estimates motion at every pixel.

OpenCV API:

```python
flow = cv2.calcOpticalFlowFarneback(prev_gray, next_gray, None,
                                    pyr_scale, levels, winsize,
                                    iterations, poly_n, poly_sigma, flags)
```

The result shape is `(H, W, 2)`:

- `flow[..., 0]`: horizontal motion `u`
- `flow[..., 1]`: vertical motion `v`

Common visualization:

- direction -> hue
- magnitude -> value / brightness

Useful when:

- you need motion segmentation
- you need a motion field, not just tracks
- you want to visualize global motion patterns

### 1.5 Failure Modes

Optical flow often fails when:

- the object has little texture
- motion between frames is too large
- lighting changes strongly
- blur removes corners
- points move out of frame
- occlusion hides the surface
- repeated patterns create ambiguous matches
- reflections and shadows violate brightness constancy
- camera motion dominates object motion

Master-level habit:

`Always inspect both the image and the flow quality signal: status flags, errors, forward-backward consistency, and motion magnitude.`

## Part 2: Coding Demo - Build Optical Flow Step by Step

### 2.1 Imports and Helpers

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

try:
    import cv2
except ImportError as exc:
    raise ImportError("OpenCV is not installed. Please run: pip install opencv-python") from exc

print("OpenCV version:", cv2.__version__)
print("Working directory:", os.getcwd())


def show_many(images, cols=3, figsize=(15, 8)):
    rows = int(np.ceil(len(images) / cols))
    plt.figure(figsize=figsize)
    for idx, (title, image, cmap) in enumerate(images, start=1):
        plt.subplot(rows, cols, idx)
        if image.ndim == 2:
            plt.imshow(image, cmap=cmap or "gray")
        else:
            plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        plt.title(title)
        plt.axis("off")
    plt.tight_layout()
    plt.show()


def draw_points(frame, points, color=(0, 255, 0), radius=4):
    vis = frame.copy()
    if points is None:
        return vis
    for x, y in np.asarray(points).reshape(-1, 2):
        cv2.circle(vis, (int(round(x)), int(round(y))), radius, color, -1, cv2.LINE_AA)
    return vis


def draw_tracks(frame, old_points, new_points, status=None, color=(0, 220, 0)):
    vis = frame.copy()
    old_points = np.asarray(old_points).reshape(-1, 2)
    new_points = np.asarray(new_points).reshape(-1, 2)
    if status is None:
        status = np.ones((len(old_points),), dtype=bool)
    else:
        status = np.asarray(status).reshape(-1).astype(bool)
    for (x0, y0), (x1, y1), ok in zip(old_points, new_points, status):
        if not ok:
            continue
        p0 = (int(round(x0)), int(round(y0)))
        p1 = (int(round(x1)), int(round(y1)))
        cv2.line(vis, p0, p1, color, 2, cv2.LINE_AA)
        cv2.circle(vis, p1, 4, (0, 0, 255), -1, cv2.LINE_AA)
    return vis


def flow_to_hsv_bgr(flow, clip_magnitude=None):
    u, v = flow[..., 0], flow[..., 1]
    magnitude, angle = cv2.cartToPolar(u, v, angleInDegrees=True)
    if clip_magnitude is None:
        clip_magnitude = np.percentile(magnitude, 99)
    clip_magnitude = max(float(clip_magnitude), 1e-6)
    hsv = np.zeros((*flow.shape[:2], 3), dtype=np.uint8)
    hsv[..., 0] = (angle / 2).astype(np.uint8)
    hsv[..., 1] = 255
    hsv[..., 2] = np.clip(magnitude / clip_magnitude * 255, 0, 255).astype(np.uint8)
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR), magnitude


def draw_flow_arrows(frame, flow, step=24, scale=1.0, color=(0, 255, 255)):
    vis = frame.copy()
    h, w = frame.shape[:2]
    for y in range(step // 2, h, step):
        for x in range(step // 2, w, step):
            dx, dy = flow[y, x]
            p0 = (x, y)
            p1 = (int(round(x + scale * dx)), int(round(y + scale * dy)))
            cv2.arrowedLine(vis, p0, p1, color, 1, cv2.LINE_AA, tipLength=0.35)
    return vis

### 2.2 Create a Synthetic Motion Sequence

We generate a small video-like sequence with texture, translation, rotation, background clutter, and mild noise. Optical flow needs texture; a perfectly flat object gives the algorithm very little to match.

In [ ]:
def make_flow_frame(t, size=(420, 640), noise_level=2.0):
    h, w = size
    frame = np.full((h, w, 3), (232, 235, 238), dtype=np.uint8)
    x_grad = np.linspace(0, 28, w, dtype=np.uint8)
    y_grad = np.linspace(0, 20, h, dtype=np.uint8)[:, None]
    frame[:, :, 0] = np.clip(frame[:, :, 0] - x_grad, 0, 255)
    frame[:, :, 1] = np.clip(frame[:, :, 1] - y_grad, 0, 255)

    cv2.rectangle(frame, (35, 55), (145, 145), (205, 215, 225), -1)
    cv2.circle(frame, (530, 92), 42, (180, 210, 190), -1)
    cv2.line(frame, (20, 360), (610, 320), (120, 120, 120), 2)
    cv2.putText(frame, "flow lab", (34, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.95, (90, 90, 90), 2, cv2.LINE_AA)

    cx = int(110 + 8.2 * t)
    cy = int(220 + 34 * np.sin(t / 7.0))
    angle = float(-14 + 1.8 * t)
    rect = ((cx, cy), (118, 82), angle)
    box = np.intp(cv2.boxPoints(rect))
    cv2.fillConvexPoly(frame, box, (42, 120, 220))
    cv2.polylines(frame, [box], True, (20, 20, 20), 2)

    patch = np.zeros_like(frame)
    for k in range(-42, 50, 18):
        cv2.line(patch, (cx - 70, cy + k), (cx + 70, cy + k + 24), (245, 245, 245), 2, cv2.LINE_AA)
    for k in range(-45, 55, 28):
        cv2.circle(patch, (cx + k, cy + int(18 * np.sin((t + k) / 6.0))), 6, (35, 35, 35), -1, cv2.LINE_AA)
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillConvexPoly(mask, box, 255)
    frame = np.where(mask[..., None] > 0, np.maximum(frame, patch), frame)

    rng = np.random.default_rng(1000 + t)
    noise = rng.normal(0, noise_level, frame.shape).astype(np.int16)
    frame = np.clip(frame.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    return frame, rect, cv2.boundingRect(box)


frames, gt_rects, gt_windows = [], [], []
for t in range(42):
    frame, rect, win = make_flow_frame(t)
    frames.append(frame)
    gt_rects.append(rect)
    gt_windows.append(win)

gray_frames = [cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) for frame in frames]

show_many(
    [("Frame 0", frames[0], None), ("Frame 12", frames[12], None),
     ("Frame 24", frames[24], None), ("Frame 36", frames[36], None)],
    cols=2,
    figsize=(12, 8),
)
print("Number of frames:", len(frames))
print("Initial ground-truth window:", gt_windows[0])

### 2.3 Detect Trackable Points

Lucas-Kanade does not decide which points are good by itself. We usually give it corners from `cv2.goodFeaturesToTrack`.

Good feature rule:

`Corners with gradients in two directions are easier to track than flat regions or long straight edges.`

In [ ]:
feature_params = dict(maxCorners=120, qualityLevel=0.02, minDistance=8, blockSize=7)
p0 = cv2.goodFeaturesToTrack(gray_frames[0], mask=None, **feature_params)
initial_points_vis = draw_points(frames[0], p0, color=(0, 255, 0), radius=4)

show_many(
    [("Frame 0", frames[0], None), ("Detected points", initial_points_vis, None)],
    cols=2,
    figsize=(12, 5),
)
print("Detected points:", 0 if p0 is None else len(p0))

### 2.4 One-Step Lucas-Kanade Flow

Track the detected points from frame 0 to frame 1. The `status` output tells which points OpenCV considers successfully tracked.

In [ ]:
lk_params = dict(
    winSize=(21, 21),
    maxLevel=3,
    criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01),
)

p1, st, err = cv2.calcOpticalFlowPyrLK(gray_frames[0], gray_frames[1], p0, None, **lk_params)
tracked_vis = draw_tracks(frames[1], p0, p1, st)

show_many(
    [("Frame 1", frames[1], None), ("LK tracks: frame 0 -> 1", tracked_vis, None)],
    cols=2,
    figsize=(12, 5),
)
print("Tracked points:", int(st.sum()), "/", len(st))
print("Mean LK error for valid points:", float(err[st == 1].mean()))

### 2.5 Track Points Through the Whole Sequence

A practical tracker repeatedly feeds the previous good points into the next frame. Points can disappear, drift, or leave the image, so the active point set usually shrinks over time.

In [ ]:
track_points = p0.copy()
status_counts = [len(track_points)]
visuals = []

for idx in range(1, len(frames)):
    next_points, status, error = cv2.calcOpticalFlowPyrLK(
        gray_frames[idx - 1], gray_frames[idx], track_points, None, **lk_params
    )
    good_new = next_points[status.reshape(-1) == 1]
    good_old = track_points[status.reshape(-1) == 1]
    if idx in [1, 10, 20, 30, 40]:
        visuals.append((f"Frame {idx}", draw_tracks(frames[idx], good_old, good_new), None))
    track_points = good_new.reshape(-1, 1, 2)
    status_counts.append(len(good_new))
    if len(track_points) < 6:
        break

show_many(visuals, cols=3, figsize=(15, 8))
plt.figure(figsize=(9, 4))
plt.plot(status_counts, linewidth=2)
plt.xlabel("Frame index")
plt.ylabel("Active tracked points")
plt.title("Sparse LK Point Survival")
plt.grid(True, alpha=0.3)
plt.show()
print("Final active points:", status_counts[-1])

### 2.6 Forward-Backward Consistency Check

A reliable point should track forward and then return near its original location when tracked backward.

`prev -> next -> prev_again`

Large round-trip error usually means the match is unreliable.

In [ ]:
def lk_forward_backward(prev_gray, next_gray, prev_points, lk_params, fb_threshold=1.5):
    next_points, status_fwd, error_fwd = cv2.calcOpticalFlowPyrLK(prev_gray, next_gray, prev_points, None, **lk_params)
    prev_again, status_bwd, error_bwd = cv2.calcOpticalFlowPyrLK(next_gray, prev_gray, next_points, None, **lk_params)
    fb_error = np.linalg.norm(prev_points.reshape(-1, 2) - prev_again.reshape(-1, 2), axis=1)
    good = (status_fwd.reshape(-1) == 1) & (status_bwd.reshape(-1) == 1) & (fb_error < fb_threshold)
    return next_points, good, fb_error, error_fwd


sample_idx = 12
p_sample = cv2.goodFeaturesToTrack(gray_frames[sample_idx], mask=None, **feature_params)
p_next, good_fb, fb_error, lk_error = lk_forward_backward(
    gray_frames[sample_idx], gray_frames[sample_idx + 1], p_sample, lk_params, fb_threshold=1.5
)

vis_all = draw_tracks(frames[sample_idx + 1], p_sample, p_next, np.ones(len(p_sample), dtype=bool), color=(0, 180, 255))
vis_good = draw_tracks(frames[sample_idx + 1], p_sample, p_next, good_fb, color=(0, 255, 0))
show_many(
    [("All LK matches", vis_all, None), ("Forward-backward filtered", vis_good, None)],
    cols=2,
    figsize=(12, 5),
)
print("All matches:", len(p_sample))
print("FB-consistent matches:", int(good_fb.sum()))
print("Median FB error:", float(np.median(fb_error)))

### 2.7 Dense Farneback Flow

Farneback gives a vector for every pixel. Here we estimate motion from frame 12 to frame 13 and visualize it as both color and arrows.

In [ ]:
farneback_params = dict(
    pyr_scale=0.5,
    levels=4,
    winsize=25,
    iterations=3,
    poly_n=7,
    poly_sigma=1.5,
    flags=0,
)

idx = 12
flow = cv2.calcOpticalFlowFarneback(gray_frames[idx], gray_frames[idx + 1], None, **farneback_params)
flow_color, magnitude = flow_to_hsv_bgr(flow)
flow_arrows = draw_flow_arrows(frames[idx + 1], flow, step=28, scale=2.5)

show_many(
    [(f"Frame {idx}", frames[idx], None), (f"Frame {idx + 1}", frames[idx + 1], None),
     ("Dense flow color", flow_color, None), ("Dense flow arrows", flow_arrows, None),
     ("Flow magnitude", magnitude, "magma")],
    cols=3,
    figsize=(15, 8),
)
print("Flow shape:", flow.shape)
print("Magnitude range:", float(magnitude.min()), "to", float(magnitude.max()))

### 2.8 Motion Mask from Dense Flow

Dense flow can become a simple motion detector by thresholding magnitude. This is not full object segmentation, but it is a useful diagnostic.

In [ ]:
mag_threshold = np.percentile(magnitude, 88)
motion_mask = (magnitude > mag_threshold).astype(np.uint8) * 255
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
motion_mask_clean = cv2.morphologyEx(motion_mask, cv2.MORPH_OPEN, kernel)
motion_mask_clean = cv2.morphologyEx(motion_mask_clean, cv2.MORPH_CLOSE, kernel)

motion_overlay = frames[idx + 1].copy()
motion_overlay[motion_mask_clean > 0] = (0.45 * motion_overlay[motion_mask_clean > 0] + np.array([0, 0, 255]) * 0.55).astype(np.uint8)

show_many(
    [("Flow magnitude", magnitude, "magma"), ("Motion mask", motion_mask_clean, "gray"),
     ("Motion overlay", motion_overlay, None)],
    cols=3,
    figsize=(15, 4),
)
print("Magnitude threshold:", float(mag_threshold))
print("Motion pixels:", int(np.count_nonzero(motion_mask_clean)))

### 2.9 Sparse vs Dense Comparison

Expected observation:

- Lucas-Kanade gives clean tracks on selected points.
- Farneback gives a full motion field, including noisy background estimates.
- Sparse flow is easier to validate per point; dense flow is easier to visualize as a field.

In [ ]:
compare_idx = 20
p_compare = cv2.goodFeaturesToTrack(gray_frames[compare_idx], mask=None, **feature_params)
p_compare_next, st_compare, err_compare = cv2.calcOpticalFlowPyrLK(
    gray_frames[compare_idx], gray_frames[compare_idx + 1], p_compare, None, **lk_params
)
flow_compare = cv2.calcOpticalFlowFarneback(
    gray_frames[compare_idx], gray_frames[compare_idx + 1], None, **farneback_params
)
flow_compare_color, flow_compare_mag = flow_to_hsv_bgr(flow_compare)

show_many(
    [("Sparse LK", draw_tracks(frames[compare_idx + 1], p_compare, p_compare_next, st_compare), None),
     ("Dense Farneback color", flow_compare_color, None),
     ("Dense Farneback arrows", draw_flow_arrows(frames[compare_idx + 1], flow_compare, step=28, scale=2.5), None)],
    cols=3,
    figsize=(15, 4),
)
print("Sparse valid points:", int(st_compare.sum()), "/", len(st_compare))
print("Dense mean magnitude:", float(flow_compare_mag.mean()))

### 2.10 Parameter Lab: Window Size and Pyramid Levels

Two important knobs:

1. Larger LK `winSize` can handle smoother/noisier patches, but may mix different motions.
2. More pyramid `maxLevel` helps larger displacement, but can increase false matches.

In [ ]:
def evaluate_lk_pair(prev_gray, next_gray, points, win_size=(21, 21), max_level=3):
    params = dict(
        winSize=win_size,
        maxLevel=max_level,
        criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01),
    )
    next_points, good, fb_error, lk_error = lk_forward_backward(prev_gray, next_gray, points, params, fb_threshold=2.0)
    valid_count = int(good.sum())
    median_fb = float(np.median(fb_error[good])) if valid_count else float("nan")
    return valid_count, median_fb, next_points, good


fast_prev, _, _ = make_flow_frame(4)
fast_next, _, _ = make_flow_frame(8)
fast_prev_gray = cv2.cvtColor(fast_prev, cv2.COLOR_BGR2GRAY)
fast_next_gray = cv2.cvtColor(fast_next, cv2.COLOR_BGR2GRAY)
fast_points = cv2.goodFeaturesToTrack(fast_prev_gray, mask=None, **feature_params)

settings = [((11, 11), 0), ((21, 21), 2), ((31, 31), 3), ((41, 41), 4)]
results = []
visuals = []
for win_size, max_level in settings:
    valid_count, median_fb, next_points, good = evaluate_lk_pair(
        fast_prev_gray, fast_next_gray, fast_points, win_size=win_size, max_level=max_level
    )
    results.append((win_size, max_level, valid_count, median_fb))
    visuals.append((f"win={win_size[0]}, level={max_level}", draw_tracks(fast_next, fast_points, next_points, good), None))

show_many(visuals, cols=2, figsize=(12, 8))
for win_size, max_level, valid_count, median_fb in results:
    print(f"winSize={win_size}, maxLevel={max_level}: valid={valid_count}, median FB error={median_fb:.3f}")

### 2.11 Reusable Optical Flow Functions

The functions below make the algorithm choice explicit and return structured records that can be plotted or consumed by a larger pipeline.

In [ ]:
def run_sparse_lk(frames, feature_params, lk_params, redetect_every=None, fb_threshold=None):
    gray = [cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) for frame in frames]
    points = cv2.goodFeaturesToTrack(gray[0], mask=None, **feature_params)
    records = []
    for idx in range(1, len(frames)):
        if points is None or len(points) == 0:
            records.append({"frame_index": idx, "old_points": np.empty((0, 2)), "new_points": np.empty((0, 2)), "good": np.empty((0,), dtype=bool)})
            if redetect_every is not None:
                points = cv2.goodFeaturesToTrack(gray[idx], mask=None, **feature_params)
            continue

        if fb_threshold is None:
            next_points, status, error = cv2.calcOpticalFlowPyrLK(gray[idx - 1], gray[idx], points, None, **lk_params)
            good = status.reshape(-1).astype(bool)
        else:
            next_points, good, fb_error, error = lk_forward_backward(gray[idx - 1], gray[idx], points, lk_params, fb_threshold=fb_threshold)

        old_good = points.reshape(-1, 2)[good]
        new_good = next_points.reshape(-1, 2)[good]
        records.append({"frame_index": idx, "old_points": old_good, "new_points": new_good, "good": good})
        points = new_good.reshape(-1, 1, 2)

        if redetect_every is not None and idx % redetect_every == 0:
            fresh = cv2.goodFeaturesToTrack(gray[idx], mask=None, **feature_params)
            if fresh is not None:
                points = np.vstack([points, fresh]).astype(np.float32)
    return records


def run_dense_farneback(frames, farneback_params):
    gray = [cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) for frame in frames]
    flows = []
    for idx in range(1, len(frames)):
        flow = cv2.calcOpticalFlowFarneback(gray[idx - 1], gray[idx], None, **farneback_params)
        flows.append({"frame_index": idx, "flow": flow})
    return flows


sparse_records = run_sparse_lk(frames, feature_params, lk_params, redetect_every=12, fb_threshold=1.8)
dense_records = run_dense_farneback(frames[:8], farneback_params)

visuals = []
for record in [sparse_records[i] for i in [0, 9, 19, 29, 39]]:
    idx = record["frame_index"]
    visuals.append((f"Frame {idx}", draw_tracks(frames[idx], record["old_points"], record["new_points"]), None))
show_many(visuals, cols=3, figsize=(15, 8))
print("Sparse records:", len(sparse_records))
print("Dense records:", len(dense_records))

### 2.12 Template for Real Video or Webcam Optical Flow

This cell prints a safe template instead of automatically opening your camera.

In [ ]:
video_optical_flow_template = r'''
import cv2
import numpy as np

cap = cv2.VideoCapture(0)  # or replace 0 with a video path
if not cap.isOpened():
    raise RuntimeError("Cannot open video source")

ok, prev_frame = cap.read()
if not ok:
    raise RuntimeError("Cannot read first frame")

prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
feature_params = dict(maxCorners=200, qualityLevel=0.02, minDistance=8, blockSize=7)
lk_params = dict(
    winSize=(21, 21),
    maxLevel=3,
    criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01),
)
points = cv2.goodFeaturesToTrack(prev_gray, mask=None, **feature_params)

while True:
    ok, frame = cap.read()
    if not ok:
        break
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    if points is None or len(points) < 20:
        points = cv2.goodFeaturesToTrack(prev_gray, mask=None, **feature_params)

    if points is not None and len(points) > 0:
        next_points, status, error = cv2.calcOpticalFlowPyrLK(prev_gray, gray, points, None, **lk_params)
        good_new = next_points[status.reshape(-1) == 1]
        good_old = points[status.reshape(-1) == 1]

        for old, new in zip(good_old.reshape(-1, 2), good_new.reshape(-1, 2)):
            x0, y0 = old
            x1, y1 = new
            cv2.line(frame, (int(x0), int(y0)), (int(x1), int(y1)), (0, 255, 0), 2)
            cv2.circle(frame, (int(x1), int(y1)), 3, (0, 0, 255), -1)
        points = good_new.reshape(-1, 1, 2)

    cv2.imshow("sparse optical flow", frame)
    if (cv2.waitKey(30) & 0xFF) == 27:
        break

    prev_gray = gray.copy()

cap.release()
cv2.destroyAllWindows()
'''
print(video_optical_flow_template)

## Part 3: Debugging and Mastery Roadmap

Debug in this order:

1. **Frame spacing**: adjacent frames should not jump too far.
2. **Texture**: corners and gradients must exist where you expect motion.
3. **Grayscale quality**: avoid overexposed, underexposed, blurred frames.
4. **Feature parameters**: `qualityLevel`, `minDistance`, and `maxCorners` control point supply.
5. **LK window**: too small misses motion; too large mixes different surfaces.
6. **Pyramid levels**: too few fails on large motion; too many may accept wrong matches.
7. **Status and error**: never treat every returned point as reliable.
8. **Forward-backward check**: reject points that do not round-trip well.
9. **Dense magnitude**: inspect whether motion appears in the expected region.
10. **Outliers**: use median motion or robust estimation before making decisions.

Ways to improve reliability:

- Redetect features periodically.
- Use a mask to detect features only inside a target region.
- Estimate an affine transform or homography from tracked points with RANSAC.
- Combine optical flow with object detection or background subtraction.
- Downsample for speed, but remember to scale vectors back if needed.
- Use frame differencing or IMU/camera metadata to diagnose large motion.

## Part 4: Practice Projects

Suggested progression:

1. **Corner motion viewer**: draw LK tracks on webcam video.
2. **Forward-backward filter lab**: compare tracks before and after consistency filtering.
3. **Dense flow heatmap**: show magnitude and direction for a moving scene.
4. **Motion segmentation**: threshold dense flow and clean it with morphology.
5. **Video stabilizer prototype**: estimate global translation from sparse flow.
6. **Object tracker hybrid**: use a detector once, then LK points inside the box.
7. **Camera vs object motion**: subtract global motion and highlight independent movers.
8. **Parameter dashboard**: interactively tune `winSize`, `maxLevel`, and Farneback settings.

## Part 5: Quick API Reference

Core OpenCV functions:

- `cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)`
- `cv2.goodFeaturesToTrack(image, maxCorners, qualityLevel, minDistance, ...)`
- `cv2.calcOpticalFlowPyrLK(prevImg, nextImg, prevPts, nextPts, ...)`
- `cv2.calcOpticalFlowFarneback(prev, next, flow, pyr_scale, levels, winsize, iterations, poly_n, poly_sigma, flags)`
- `cv2.cartToPolar(x, y, angleInDegrees=True)`
- `cv2.arrowedLine(image, pt1, pt2, color, thickness)`
- `cv2.circle(image, center, radius, color, thickness)`
- `cv2.line(image, pt1, pt2, color, thickness)`
- `cv2.morphologyEx(src, op, kernel)`
- `cv2.VideoCapture(source)`

Parameter reminders:

- LK `winSize`: local matching window size.
- LK `maxLevel`: pyramid depth for larger motion.
- LK `criteria`: iteration stopping rule.
- Farneback `winsize`: averaging window size.
- Farneback `levels`: pyramid levels.
- Farneback `poly_n` and `poly_sigma`: local polynomial model smoothness.

## Conclusion

Optical flow teaches one of the most important ideas in video understanding:

`motion is evidence, not truth`

Lucas-Kanade gives fast, inspectable point tracks. Farneback gives a dense motion field that is excellent for visualization and motion masks. Once you understand where optical flow is reliable and where it lies, tracking, stabilization, motion segmentation, and video analysis become much easier to reason about.